In [6]:
# === 山峰最近C0氣象站精確計算 ===
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, asin

def distance_haversine(lat1, lon1, lat2, lon2):
    """精確 Haversine 距離計算（公里）"""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 6371 * 2 * asin(sqrt(a))

# 載入資料
mountains = pd.read_csv('./csv/mountain_lanlon.csv').dropna()
stations = pd.read_csv('./csv/observation_stations.csv')

# 篩選 C0 開頭的氣象站並清理
c0_stations = stations[stations['站號'].str.startswith('C0', na=False)].copy()
c0_stations = c0_stations.dropna(subset=['經度', '緯度'])
c0_stations[['經度', '緯度']] = c0_stations[['經度', '緯度']].apply(pd.to_numeric, errors='coerce')
c0_stations = c0_stations.dropna(subset=['經度', '緯度'])

# *** 關鍵修正：重置索引 ***
c0_stations = c0_stations.reset_index(drop=True)

print(f"山峰數量: {len(mountains)}")
print(f"C0氣象站數量: {len(c0_stations)}")

# 計算結果
results = []

for _, mountain in mountains.iterrows():
    if pd.notna(mountain['山名']) and mountain['山名'].strip():
        mountain_name = mountain['山名']
        mountain_lat = mountain['緯度']
        mountain_lon = mountain['經度']
        
        min_distance = float('inf')
        nearest_station = None
        
        # 對每個C0氣象站計算距離
        for idx, station in c0_stations.iterrows():
            distance = distance_haversine(
                mountain_lat, mountain_lon,
                station['緯度'], station['經度']
            )
            
            if distance < min_distance:
                min_distance = distance
                nearest_station = station
        
        result = {
            'mountain_name': mountain_name,
            'station_name': nearest_station['站名'],
            'station_id': nearest_station['站號'],
            'distance_km': round(min_distance, 3),
            'mountain_lat': mountain_lat,
            'mountain_lon': mountain_lon,
            'station_lat': nearest_station['緯度'],
            'station_lon': nearest_station['經度'],
            'station_elevation': nearest_station['海拔高度(m)'],
            'station_city': nearest_station['城市']
        }
        
        results.append(result)
        print(f"{mountain_name:15s} -> {nearest_station['站名']:15s} ({min_distance:6.3f} km)")

# 轉換為 DataFrame
df_results = pd.DataFrame(results)

# 保存結果
df_results.to_csv('csv/mountain_detailed.csv', index=False, encoding='utf-8-sig')

# 簡化版結果
simple = df_results[['mountain_name', 'station_name', 'station_id', 'distance_km']].copy()
simple.columns = ['山名', '最近氣象站', '站號', '距離(公里)']
simple.to_csv('csv/mountain_simple.csv', index=False, encoding='utf-8-sig')

# 統計資訊
print(f"\n統計:")
print(f"處理山峰: {len(df_results)} 座")
print(f"平均距離: {df_results['distance_km'].mean():.3f} km")
print(f"最小距離: {df_results['distance_km'].min():.3f} km")
print(f"最大距離: {df_results['distance_km'].max():.3f} km")

print("\n結果已保存:")

山峰數量: 23
C0氣象站數量: 504
奇萊主峰            -> 奇萊稜線            ( 2.604 km)
奇萊北峰            -> 奇萊稜線            ( 1.288 km)
桃山              -> 桃山              ( 0.112 km)
喀拉業山            -> 桃山              ( 2.598 km)
合歡山北峰           -> 大禹嶺             ( 3.536 km)
合歡山北峰西北峰        -> 大禹嶺             ( 3.536 km)
白姑大山            -> 中橫21.6k         ( 6.474 km)
北大武山            -> 瑪家              ( 9.867 km)
志佳陽大山           -> 雪山圈谷            ( 3.406 km)
羊頭山             -> 大禹嶺             ( 6.913 km)
畢祿山             -> 大禹嶺             ( 4.393 km)
西巒大山            -> 信義              ( 9.823 km)
雪山東峰            -> 雪山東峰            ( 0.337 km)
塔關山             -> 向陽              ( 4.587 km)
關山嶺山            -> 向陽              ( 3.704 km)
屏風山             -> 奇萊稜線            ( 4.751 km)
品田山             -> 桃山              ( 0.940 km)
池有山             -> 桃山              ( 1.023 km)
小關山             -> 復興              ( 6.460 km)
郡大山             -> 玉山風口            (19.741 km)
玉山前峰            -> 玉山風口            ( 1